# Task 3: Forecast Future Market Trends

## Objective
Use trained models to forecast Tesla's future stock prices and analyze the results for actionable insights. Generate future predictions, visualize them with uncertainty bounds, and translate results into business insights.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

# Load models
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler

plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load Models and Historical Data

In [ ]:
# Load TSLA data
tsla_data = pd.read_csv('../data/processed/tsla_processed.csv', index_col=0, parse_dates=True)
tsla_data = tsla_data.sort_index()

# Load model information
with open('../data/processed/model_info.json', 'r') as f:
    model_info = json.load(f)

best_model_name = model_info['Best_Model']
print(f"Best Model: {best_model_name}")

# Load the best model
if best_model_name == "ARIMA/SARIMA":
    with open('../data/processed/arima_model.pkl', 'rb') as f:
        best_model = pickle.load(f)
    model_type = 'arima'
else:
    best_model = load_model('../data/processed/lstm_model.h5')
    with open('../data/processed/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    model_type = 'lstm'

print(f"Model loaded successfully!")
print(f"Historical data shape: {tsla_data.shape}")

## 2. Generate Future Forecasts (6-12 months)

In [ ]:
# Define forecast horizon (12 months = ~252 trading days)
forecast_horizon = 252
last_date = tsla_data.index.max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_horizon, freq='B')

print(f"Forecast period: {future_dates[0]} to {future_dates[-1]}")
print(f"Forecast horizon: {forecast_horizon} trading days (~12 months)")

# Get historical closing prices for context
historical_prices = tsla_data['Close']

In [ ]:
# Generate forecasts based on model type
if model_type == 'arima':
    # ARIMA/SARIMA forecast with confidence intervals
    forecast_result = best_model.get_forecast(steps=forecast_horizon)
    future_forecast = forecast_result.predicted_mean
    conf_int = forecast_result.conf_int()
    lower_bound = conf_int.iloc[:, 0]
    upper_bound = conf_int.iloc[:, 1]
    
else:
    # LSTM forecast (iterative)
    window_size = model_info['LSTM_Window_Size']
    last_sequence = scaler.transform(historical_prices[-window_size:].values.reshape(-1, 1)).flatten()
    
    def lstm_multi_step_forecast(model, last_sequence, n_steps):
        forecasts = []
        current_seq = last_sequence.copy()
        
        for _ in range(n_steps):
            next_pred = model.predict(current_seq.reshape(1, window_size, 1), verbose=0)
            forecasts.append(next_pred[0, 0])
            current_seq = np.append(current_seq[1:], next_pred[0, 0])
        
        return np.array(forecasts)
    
    # Generate forecast
    forecast_scaled = lstm_multi_step_forecast(best_model, last_sequence, forecast_horizon)
    future_forecast = scaler.inverse_transform(forecast_scaled.reshape(-1, 1)).flatten()
    
    # For LSTM, estimate confidence intervals using historical prediction errors
    # Load test predictions to estimate error distribution
    try:
        test_forecasts = pd.read_csv('../data/processed/model_forecasts.csv')
        test_errors = test_forecasts['Actual'] - test_forecasts[f'{best_model_name}_Forecast']
        error_std = test_errors.std()
        
        # Create confidence intervals (assuming normal distribution)
        lower_bound = future_forecast - 1.96 * error_std
        upper_bound = future_forecast + 1.96 * error_std
    except:
        # Fallback: use percentage-based intervals
        error_pct = 0.05  # 5% error estimate
        lower_bound = future_forecast * (1 - error_pct)
        upper_bound = future_forecast * (1 + error_pct)

# Create forecast dataframe
forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Forecast': future_forecast,
    'Lower_Bound': lower_bound,
    'Upper_Bound': upper_bound
})
forecast_df.set_index('Date', inplace=True)

print(f"\nForecast Summary:")
print(f"Initial forecast: ${forecast_df['Forecast'].iloc[0]:.2f}")
print(f"Final forecast: ${forecast_df['Forecast'].iloc[-1]:.2f}")
print(f"Expected change: {((forecast_df['Forecast'].iloc[-1] / forecast_df['Forecast'].iloc[0]) - 1) * 100:.2f}%")

## 3. Visualize Forecasts with Confidence Intervals

In [ ]:
# Create comprehensive forecast visualization
fig, ax = plt.subplots(figsize=(16, 8))

# Plot historical data (last 2 years for context)
historical_recent = historical_prices[historical_prices.index >= historical_prices.index.max() - pd.DateOffset(years=2)]
ax.plot(historical_recent.index, historical_recent.values, 
        label='Historical Prices', color='blue', linewidth=2)

# Plot forecast
ax.plot(forecast_df.index, forecast_df['Forecast'], 
        label='Forecast', color='red', linewidth=2, linestyle='--')

# Plot confidence intervals
ax.fill_between(forecast_df.index, forecast_df['Lower_Bound'], forecast_df['Upper_Bound'],
                alpha=0.3, color='red', label='95% Confidence Interval')

# Add vertical line at forecast start
ax.axvline(x=last_date, color='green', linestyle=':', linewidth=2, label='Forecast Start')

ax.set_title(f'TSLA Stock Price Forecast - Next 12 Months ({best_model_name})', 
             fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price (USD)', fontsize=12)
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/future_forecast.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Trend Analysis

In [ ]:
# Analyze forecast trends
forecast_returns = forecast_df['Forecast'].pct_change().dropna() * 100

# Calculate rolling statistics
forecast_df['Rolling_Mean_30'] = forecast_df['Forecast'].rolling(window=30).mean()
forecast_df['Rolling_Std_30'] = forecast_df['Forecast'].rolling(window=30).std()

# Identify trend direction
initial_price = forecast_df['Forecast'].iloc[0]
final_price = forecast_df['Forecast'].iloc[-1]
price_change = final_price - initial_price
price_change_pct = (price_change / initial_price) * 100

# Calculate confidence interval width over time
forecast_df['CI_Width'] = forecast_df['Upper_Bound'] - forecast_df['Lower_Bound']
forecast_df['CI_Width_Pct'] = (forecast_df['CI_Width'] / forecast_df['Forecast']) * 100

print("="*60)
print("TREND ANALYSIS")
print("="*60)
print(f"\nOverall Trend:")
print(f"  Initial Forecast: ${initial_price:.2f}")
print(f"  Final Forecast: ${final_price:.2f}")
print(f"  Expected Change: ${price_change:.2f} ({price_change_pct:.2f}%)")

if price_change_pct > 5:
    trend_direction = "BULLISH (Upward)"
elif price_change_pct < -5:
    trend_direction = "BEARISH (Downward)"
else:
    trend_direction = "NEUTRAL (Sideways)"

print(f"  Trend Direction: {trend_direction}")

print(f"\nVolatility Analysis:")
print(f"  Average Daily Return: {forecast_returns.mean():.2f}%")
print(f"  Daily Return Std: {forecast_returns.std():.2f}%")
print(f"  Max Daily Gain: {forecast_returns.max():.2f}%")
print(f"  Max Daily Loss: {forecast_returns.min():.2f}%")

print(f"\nConfidence Interval Analysis:")
print(f"  Initial CI Width: ${forecast_df['CI_Width'].iloc[0]:.2f} ({forecast_df['CI_Width_Pct'].iloc[0]:.2f}%)")
print(f"  Final CI Width: ${forecast_df['CI_Width'].iloc[-1]:.2f} ({forecast_df['CI_Width_Pct'].iloc[-1]:.2f}%)")
print(f"  CI Width Change: {((forecast_df['CI_Width'].iloc[-1] / forecast_df['CI_Width'].iloc[0]) - 1) * 100:.2f}%")

In [ ]:
# Visualize confidence interval width over time
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Plot 1: Forecast with CI width
axes[0].plot(forecast_df.index, forecast_df['Forecast'], 
             label='Forecast', color='red', linewidth=2)
axes[0].fill_between(forecast_df.index, forecast_df['Lower_Bound'], forecast_df['Upper_Bound'],
                     alpha=0.3, color='red', label='95% Confidence Interval')
axes[0].set_title('Forecast with Confidence Intervals', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price (USD)', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Confidence interval width over time
axes[1].plot(forecast_df.index, forecast_df['CI_Width_Pct'], 
             color='purple', linewidth=2)
axes[1].set_title('Confidence Interval Width Over Time (as % of Forecast)', 
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('CI Width (%)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/forecast_uncertainty_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Market Opportunities and Risks Assessment

In [ ]:
# Assess opportunities and risks
current_price = historical_prices.iloc[-1]
forecast_3m = forecast_df['Forecast'].iloc[63]  # ~3 months
forecast_6m = forecast_df['Forecast'].iloc[126]  # ~6 months
forecast_12m = forecast_df['Forecast'].iloc[-1]  # 12 months

opportunities = []
risks = []

# Opportunities
if forecast_12m > current_price * 1.1:  # >10% increase
    opportunities.append(f"Strong upward trend expected: {((forecast_12m/current_price)-1)*100:.1f}% potential gain over 12 months")
if forecast_3m > current_price:
    opportunities.append(f"Short-term positive momentum: {((forecast_3m/current_price)-1)*100:.1f}% expected in 3 months")
if forecast_df['Forecast'].min() > current_price * 0.9:  # Never drops more than 10%
    opportunities.append("Limited downside risk: Forecast suggests price stability")

# Risks
if forecast_12m < current_price * 0.9:  # >10% decrease
    risks.append(f"Potential decline: {((forecast_12m/current_price)-1)*100:.1f}% expected drop over 12 months")
if forecast_df['CI_Width_Pct'].mean() > 15:  # High uncertainty
    risks.append(f"High forecast uncertainty: Average CI width of {forecast_df['CI_Width_Pct'].mean():.1f}% indicates low confidence")
if forecast_returns.std() > 3:  # High volatility
    risks.append(f"High volatility expected: Daily return std of {forecast_returns.std():.2f}% indicates significant price swings")

# Check for widening confidence intervals
ci_widening = (forecast_df['CI_Width_Pct'].iloc[-1] / forecast_df['CI_Width_Pct'].iloc[0] - 1) * 100
if ci_widening > 20:
    risks.append(f"Forecast reliability decreases over time: CI width increases by {ci_widening:.1f}%")

print("="*60)
print("MARKET OPPORTUNITIES")
print("="*60)
if opportunities:
    for i, opp in enumerate(opportunities, 1):
        print(f"{i}. {opp}")
else:
    print("No significant opportunities identified based on forecast.")

print("\n" + "="*60)
print("MARKET RISKS")
print("="*60)
if risks:
    for i, risk in enumerate(risks, 1):
        print(f"{i}. {risk}")
else:
    print("No significant risks identified based on forecast.")

print("\n" + "="*60)
print("FORECAST RELIABILITY ASSESSMENT")
print("="*60)
print(f"\nShort-term (1-3 months):")
print(f"  CI Width: {forecast_df['CI_Width_Pct'].iloc[:63].mean():.2f}% - ", end="")
if forecast_df['CI_Width_Pct'].iloc[:63].mean() < 10:
    print("HIGH reliability")
elif forecast_df['CI_Width_Pct'].iloc[:63].mean() < 20:
    print("MODERATE reliability")
else:
    print("LOW reliability")

print(f"\nMedium-term (3-6 months):")
print(f"  CI Width: {forecast_df['CI_Width_Pct'].iloc[63:126].mean():.2f}% - ", end="")
if forecast_df['CI_Width_Pct'].iloc[63:126].mean() < 10:
    print("HIGH reliability")
elif forecast_df['CI_Width_Pct'].iloc[63:126].mean() < 20:
    print("MODERATE reliability")
else:
    print("LOW reliability")

print(f"\nLong-term (6-12 months):")
print(f"  CI Width: {forecast_df['CI_Width_Pct'].iloc[126:].mean():.2f}% - ", end="")
if forecast_df['CI_Width_Pct'].iloc[126:].mean() < 10:
    print("HIGH reliability")
elif forecast_df['CI_Width_Pct'].iloc[126:].mean() < 20:
    print("MODERATE reliability")
else:
    print("LOW reliability")

print(f"\nConclusion: Forecast reliability {'decreases' if ci_widening > 0 else 'remains stable or improves'} over time.")
print(f"This is expected as uncertainty compounds with longer forecast horizons.")

## 6. Save Forecast Results

In [ ]:
# Save forecast results
forecast_df.to_csv('../data/processed/future_forecast_12m.csv')

# Save summary
summary = {
    'Forecast_Horizon_Months': 12,
    'Forecast_Horizon_Days': forecast_horizon,
    'Current_Price': float(current_price),
    'Forecast_3M': float(forecast_3m),
    'Forecast_6M': float(forecast_6m),
    'Forecast_12M': float(forecast_12m),
    'Expected_Change_12M_Pct': float(price_change_pct),
    'Trend_Direction': trend_direction,
    'Average_CI_Width_Pct': float(forecast_df['CI_Width_Pct'].mean()),
    'Opportunities': opportunities,
    'Risks': risks
}

with open('../data/processed/forecast_summary.json', 'w') as f:
    json.dump(summary, f, indent=4)

print("Forecast results saved successfully!")
print(f"\nSummary saved to: ../data/processed/forecast_summary.json")

## Summary

### Key Insights

1. **Forecast Trend**: The model predicts [BULLISH/BEARISH/NEUTRAL] trend over the next 12 months
2. **Confidence Intervals**: The width of confidence intervals [increases/decreases] over time, indicating [higher/lower] uncertainty for longer-term forecasts
3. **Market Opportunities**: [List key opportunities]
4. **Market Risks**: [List key risks]
5. **Forecast Reliability**: Short-term forecasts are more reliable than long-term forecasts, which is consistent with time series forecasting principles